# <font color=firebrick>Econometrics — M2 MBFA</font>
## Case study — Do retirement savings plans make households richer?

**Prerequisites:** Sessions 1-6 (CM) and TD 1-3.

**What this case study is.** In the final assessment, you will carry out an empirical study on your own. This notebook shows, on one complete example, what such a study looks like: a question, a first naive regression, then a sequence of improvements, each one motivated by a weakness of the previous model. At every step, the question is the same: *what does this estimate really measure and what could make it wrong?*

**How to work.** Each step follows the same pattern:
1. run or complete the code;
2. write your interpretation in the answer cell before looking at the solution;
3. click *Show expected results* to compare with the numbers and your interpretation. The data are fixed, so your numbers should match exactly.

**Objectives:**
- Go from a raw correlation to a credible estimate, one justified step at a time.
- Choose the model according to the variables: OLS for a continuous outcome, probit/logit for a binary one, IV for an endogenous regressor.
- Read each new result *in the light of the previous one*: why did the coefficient move?

| Step | Question | Tool (session) |
|---|---|---|
| 0 | What are we studying? | Descriptive statistics (TD 1) |
| 1 | Do participants have more wealth? | Simple OLS on a dummy (Sessions 1-2) |
| 2 | Is it just income? | Omitted variable bias (Session 3) |
| 3 | Is the model well specified? | Quadratic terms, more controls, $F$-test (Session 3) |
| 4 | Is the effect the same for everyone? | Interaction (Session 3) |
| 5 | Is the result driven by a few very rich households? | Robustness to outliers (TD 1) |
| 6 | Who chooses to participate? | LPM, probit, logit, marginal effects (Session 4) |
| 7 | Is participation itself endogenous? | Instrumental variables, 2SLS (Session 6) |
| 8 | What do we conclude? | Synthesis |

---

## Step 0 — The question and the data

### 0.1 The economic question

A 401(k) plan is a retirement savings account offered by US employers. Contributions are deducted from the employee's pay *before tax*, and the employer often adds a matching contribution. Two institutional facts matter for the econometrics:

- **Eligibility is decided by the employer**: not all firms offer a plan, so not all workers can participate.
- **Participation is a choice of the eligible employee**: about 30% of eligible workers do not participate.

In the 1990s, these plans were at the centre of a policy debate: do they create new saving or do households simply move into a tax-advantaged account the savings they would have made anyway? Poterba, Venti and Wise (1995) argued for mostly new saving; Engen, Gale and Scholz (1996) for mostly reshuffling. The same question arises today for any tax-advantaged savings product (in France, the PER or the *assurance-vie*).

Our question: does participating in a 401(k) plan increase a household's net financial assets, and by how much?

Before investigating the data, form an expectation. Participants accumulate assets inside the plan, so we expect a positive association. But if participants are households that would have saved a lot anyway, the *causal* effect is smaller than the raw gap. Keep this tension in mind: the whole case study is about it.

### 0.2 The data

We use `401ksubs` from the `wooldridge` package: $9\,275$ US households from the 1991 *Survey of Income and Program Participation* (data from Abadie, 2003). The main variables:

| Variable | Meaning |
|---|---|
| `nettfa` | net total financial assets, thousand dollars (the outcome $y$) |
| `p401k` | $=1$ if participates in a 401(k) plan (the variable of interest) |
| `e401k` | $=1$ if eligible for a 401(k) plan |
| `inc` | annual family income, thousand dollars |
| `age` | age of the respondent, years (25 to 64) |
| `marr`, `male`, `fsize` | $=1$ if married; $=1$ if male respondent; family size |
| `pira` | $=1$ if the household has an IRA (another retirement account) |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from wooldridge import load_data

df = load_data.data('401ksubs', description=True)

In [ ]:
df[['nettfa', 'p401k', 'e401k', 'inc', 'age', 'marr', 'male', 'fsize']].describe().T.round(2)

**Task 0.1.** Run the cell below, then answer:
1. Compare the mean and the median of `nettfa`. What does the gap tell you about the shape of the distribution?
2. What share of households has negative or zero net financial assets? Could we use $\log(\text{nettfa})$ as the dependent variable, as we did for salaries in TD 1?
3. What consequence could the shape of this distribution have for OLS estimates?

In [ ]:
print(df['nettfa'].quantile([0.01, 0.25, 0.5, 0.75, 0.99]).round(1))
print("Share with nettfa < 0: ", round((df['nettfa'] < 0).mean(), 3))
print("Share with nettfa = 0: ", round((df['nettfa'] == 0).mean(), 3))

top1 = df['nettfa'] >= df['nettfa'].quantile(0.99)
print("Share of total wealth held by the top 1%:", round(df.loc[top1, 'nettfa'].sum() / df['nettfa'].sum(), 3))

df['nettfa'].clip(-50, 300).hist(bins=70)
plt.xlabel('nettfa (thousand dollars, clipped to [-50, 300] for readability)')
plt.ylabel('Number of households')
plt.show()

> *Your answer:* ...

<details>
<summary>Show answers</summary>

1. The mean is $19.1$ thousand dollars, the median only $2.0$: the distribution is **strongly right-skewed**. Half of the households have almost no financial assets, while a few have a lot (maximum $1\,537$). The top 1% hold about **24%** of all the financial wealth in the sample.
2. About **29%** of households have *negative* net financial assets (more debt than financial assets) and 6% exactly zero. The logarithm is not defined for these values, so we keep `nettfa` **in levels**: every coefficient will read in thousand dollars.
3. OLS minimizes *squared* residuals, so a few very large values can weigh heavily on the estimates and inflate their standard errors. We will check this in Step 5.

</details>

---

## Step 1 — A first look: participants versus non-participants

**Task 1.1.** Run the cell below: it compares the average characteristics of participants and non-participants.

In [ ]:
df.groupby('p401k')[['nettfa', 'inc', 'age', 'marr', 'fsize']].mean().round(2)

**Task 1.2.** Estimate the simple regression $\text{nettfa}_i=\beta_0+\beta_1\,\text{p401k}_i+\varepsilon_i$, then complete the code to check that $\hat\beta_0$ and $\hat\beta_1$ can be read directly from the table above.

In [ ]:
m1 = smf.ols('nettfa ~ p401k', data=df).fit()
print(m1.summary().tables[1])

# TODO: compute the mean of nettfa for non-participants and for participants
mean_non_part = ...
mean_part = ...

assert np.isclose(m1.params['Intercept'], mean_non_part), "The intercept should equal the mean of non-participants."
assert np.isclose(m1.params['p401k'], mean_part - mean_non_part), "The slope should equal the difference in means."
print("Self-check passed!")

**Questions.**
1. Interpret $\hat\beta_0$ and $\hat\beta_1$ in thousand dollars. Is $\hat\beta_1$ statistically significant?
2. Why is the regression on a single dummy variable just a comparison of two means?
3. Can we conclude that participating in a 401(k) plan raises a household's financial wealth by about 27 thousand dollars? Use the table of Task 1.1.

> *Your answer:* ...

<details>
<summary>Show expected results</summary>

1. $\hat\beta_0=11.67$: non-participants hold on average $11.7$ thousand dollars of net financial assets. $\hat\beta_1=26.81$ (standard error $1.46$, $t=18.4$): participants hold on average $26.8$ thousand dollars **more**. The effect is highly significant. $R^2=0.035$: participation alone explains very little of the differences in wealth between households.
2. With a dummy, the model predicts $\beta_0$ when $\text{p401k}=0$ and $\beta_0+\beta_1$ when $\text{p401k}=1$. OLS chooses these two values to fit each group as well as possible, which gives each group's mean.
3. **No.** The table shows that participants are not like non-participants: their income is much higher ($49.8$ against $35.2$ thousand dollars). Income raises wealth directly (more resources to save) *and* is associated with participation (higher-paying firms offer plans more often, richer employees can afford to contribute). Part of the $26.8$ is an income effect, not a 401(k) effect: this is the omitted variable bias of Session 3.

</details>

### <font color='teal'>Quick check #1</font>

**Let $\text{inc}$ be omitted from the simple regression. Income raises wealth ($\beta_{\text{inc}}>0$) and participants have higher incomes ($\delta>0$, where $\delta$ is the slope of $\text{inc}$ on $\text{p401k}$). What is the direction of the bias of $\hat\beta_1=26.81$?**

a) No bias: income is not in the model.<br>
b) Upward bias: $\hat\beta_1$ overstates the effect of participation.<br>
c) Downward bias: $\hat\beta_1$ understates the effect of participation.<br>
d) The direction cannot be determined.

<details>
<summary>Show answer</summary>

Answer: **b)**. The bias term is $\beta_{\text{inc}}\,\delta$, the product of two positive numbers. The simple regression attributes to participation part of the wealth that is really due to income.

</details>

---

## Step 2 — Controlling for income

**Task 2.1.** Add income to the regression: $\text{nettfa}_i=\beta_0+\beta_1\,\text{p401k}_i+\beta_2\,\text{inc}_i+\varepsilon_i$.

In [ ]:
m2 = smf.ols('nettfa ~ p401k + inc', data=df).fit()
print(m2.summary().tables[1])
print("R-squared:", round(m2.rsquared, 3))

**Task 2.2.** Session 3 (and TD 2) showed that the simple-regression coefficient decomposes *exactly* as $\tilde\beta_1=\hat\beta_1+\hat\beta_2\,\tilde\delta$, where $\tilde\delta$ is the slope of the regression of the omitted variable (`inc`) on the included one (`p401k`). Complete the code to check it on these data.

In [ ]:
# TODO: estimate delta, the slope of the regression of inc on p401k
delta = ...

# TODO: rebuild the simple-regression coefficient from m2's coefficients and delta
beta1_tilde = ...

print("delta       =", round(delta, 3))
print("Rebuilt     =", round(beta1_tilde, 3))
print("Step 1 (m1) =", round(m1.params['p401k'], 3))
assert np.isclose(beta1_tilde, m1.params['p401k']), "The decomposition should hold exactly - check delta and the formula."
print("Self-check passed!")

**Questions.**
1. How did the coefficient on `p401k` change between Step 1 and Step 2? Interpret the coefficient on `inc`.
2. Use the decomposition to explain *where* the difference comes from.
3. Does controlling for income settle the question? What other factors might differ between participants and non-participants?

> *Your answer:* ...

<details>
<summary>Show expected results</summary>

1. The coefficient on `p401k` falls from $26.81$ to $13.18$ (standard error $1.42$): **half** of the raw gap disappears once income is held constant. For two households with the same income, participants hold on average $13.2$ thousand dollars more. The coefficient on `inc`, $0.934$, means that one thousand dollars more of annual income is associated with $934$ dollars more of financial assets, at a given participation status. $R^2$ goes from $0.035$ to $0.150$.
2. $\tilde\delta=14.59$: participants earn on average $14.6$ thousand dollars more. Then $13.18+0.934\times14.59=26.81$. The difference between the two estimates is exactly the "income channel": participants' extra income ($14.59$) times the effect of income on wealth ($0.934$).
3. No. Income was the most obvious difference, but there are others: **age** (wealth accumulates over the life cycle), family situation, and, above all, characteristics we do not observe, such as the **taste for saving**, patience or financial literacy. Observed ones can be added as controls (Step 3). Unobserved ones cannot, and will lead us to instrumental variables (Step 7).

</details>

---

## Step 3 — Functional form and additional controls

Step 2 assumed that wealth increases *linearly* with income. Before adding variables, look at the data.

**Task 3.1.** The cell below plots the average `nettfa` within each decile of income, and within each 5-year age group. Is the relationship linear?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

df.groupby(pd.qcut(df['inc'], 10), observed=True)['nettfa'].mean().plot(ax=axes[0], marker='o', rot=45)
axes[0].set_title('Mean nettfa by income decile')
axes[0].set_ylabel('nettfa (thousand dollars)')

df.groupby(pd.cut(df['age'], range(25, 70, 5), right=False), observed=True)['nettfa'].mean().plot(ax=axes[1], marker='o', rot=45)
axes[1].set_title('Mean nettfa by age group')

plt.tight_layout()
plt.show()

**Task 3.2.** Wealth rises faster and faster with income (a clearly convex curve), and steadily with age, with some curvature as well. A straight line fits neither. Estimate a model that allows for this curvature (quadratic terms, Session 3) and adds the demographic controls:
$$
\text{nettfa}_i=\beta_0+\beta_1\,\text{p401k}_i+\beta_2\,\text{inc}_i+\beta_3\,\text{inc}_i^2+\beta_4\,\text{age}_i+\beta_5\,\text{age}_i^2+\beta_6\,\text{marr}_i+\beta_7\,\text{fsize}_i+\beta_8\,\text{male}_i+\varepsilon_i.
$$
In a formula, a squared term is written `I(inc**2)`.

In [ ]:
# TODO: write the list of controls (everything except p401k), using I(...) for the squared terms
controls = "..."

m3 = smf.ols('nettfa ~ p401k + ' + controls, data=df).fit()
print(m3.summary().tables[1])
print("R-squared:", round(m3.rsquared, 3))
assert np.isclose(m3.params['p401k'], 15.84, atol=0.01), "Check your list of controls."
print("Self-check passed!")

**Task 3.3.** With a quadratic term, the effect of age is $\beta_4+2\beta_5\,\text{age}$, which changes sign at $\text{age}^*=-\beta_4/(2\beta_5)$. Compute this turning point. Then test whether the three demographic controls are jointly significant.

In [ ]:
# TODO: compute the age at which the effect of age on wealth changes sign
age_star = ...
print("Turning point:", round(age_star, 1), "years")
assert np.isclose(age_star, 26.6, atol=0.1), "Use -beta_age / (2 * beta_age_squared)."
print("Self-check passed!")

print(m3.f_test('marr = 0, fsize = 0, male = 0'))

**Questions.**
1. The coefficient on `inc` is now *negative*. Does this mean that richer households have fewer financial assets? Compute the effect of one more thousand dollars of income at the mean income ($39.3$) and at $80$.
2. Interpret the turning point for age, given that the sample covers ages 25 to 64.
3. `male` is not individually significant. Should we drop the three demographic variables?
4. How has the coefficient on `p401k` evolved since Step 2? Which change mattered most for this coefficient: controlling for income at all (Step 2), or refining *how* we control for it (Step 3)?

> *Your answer:* ...

<details>
<summary>Show expected results</summary>

1. No: with a quadratic term, the coefficient on `inc` alone is the effect at $\text{inc}=0$, which is outside the data. The effect of income is $-0.240+2\times0.0099\times\text{inc}$: about $+0.54$ at the mean income and $+1.34$ at $80$ thousand dollars. Each extra thousand dollars of income adds more and more to financial wealth: higher-income households save a larger share of their income.
2. $\text{age}^*\approx26.6$ years, close to the minimum age of the sample. Over almost the whole sample, wealth therefore **increases** with age, and at an increasing rate. For a married, non-participating household of 3 with mean income, predicted assets are about $-2$ thousand dollars at 25, $0$ at 35, $8$ at 45, $23$ at 55 and $41$ at 64. This is **life-cycle saving**: households accumulate as retirement approaches.
3. No. The joint test gives $F=8.24$ ($p<0.001$): the three variables are jointly significant, mainly thanks to `fsize` (larger families hold fewer financial assets, $-1.27$ per member). A variable that is not individually significant can be kept as a control, especially when the joint test rejects.
4. The coefficient went from $13.18$ to $15.84$ (standard error $1.38$), and $R^2$ from $0.150$ to $0.211$:

| Model | `p401k` | $R^2$ |
|---|---|---|
| Step 1: no control | $26.81$ | $0.035$ |
| Step 2: + `inc` | $13.18$ | $0.150$ |
| + `inc`² | $15.65$ | $0.177$ |
| + `age`, `age`² | $16.07$ | $0.208$ |
| Step 3: + `marr`, `fsize`, `male` | $15.84$ | $0.211$ |

The big change was controlling for income at all ($27\to13$). Refining the functional form moves the estimate by about $2.5$ thousand dollars, since participants and non-participants are now compared at the same income *along a curve* rather than along a straight line. The demographic controls barely change it. Since they matter for wealth but are not very correlated with participation, they cause little bias when omitted.

</details>

---

## Step 4 — Is the effect the same for everyone?

So far, the model imposes the same effect of participation on every household. Contributions are a percentage of pay, and the tax advantage grows with the marginal tax rate, so we might expect the plan to matter more for higher incomes.

**Task 4.1.** Add an interaction between participation and income (Session 3): `p401k:inc`. The effect of participation is then $\beta_1+\beta_{\text{int}}\times\text{inc}$. Compute it at incomes of $20$, $39.3$ (the mean) and $80$ thousand dollars.

In [ ]:
m4 = smf.ols('nettfa ~ p401k + p401k:inc + ' + controls, data=df).fit()
print(m4.params[['p401k', 'p401k:inc']].round(3))
print("p-value of the interaction:", round(m4.pvalues['p401k:inc'], 4))

# TODO: compute the effect of participation at each income level
incomes = np.array([20, 39.3, 80])
effects = ...

print(dict(zip(incomes, effects.round(1))))
assert np.allclose(effects, [12.0, 15.0, 21.1], atol=0.1), "Effect = coefficient of p401k + coefficient of p401k:inc * income."
print("Self-check passed!")

**Questions.**
1. Is the effect of participation heterogeneous? Interpret.
2. The coefficient on `p401k` alone dropped to about $9$. Is this the "new" effect of participation? How could you make this coefficient directly interpretable? (Think of Option B of the final assessment.)

> *Your answer:* ...

<details>
<summary>Show expected results</summary>

1. Yes. The interaction coefficient is $0.151$ ($p=0.006$). The effect of participation is about $12.0$ thousand dollars at an income of $20$, $15.0$ at the mean income, and $21.1$ at $80$. Participation is associated with a larger wealth gain for higher-income households, consistent with contributions proportional to pay and a larger tax advantage. Note that at the mean income we recover roughly the Step 3 estimate ($15.8$).
2. No. With an interaction, the coefficient on `p401k` alone ($9.0$) is the effect at $\text{inc}=0$, which no household has. **Centering** income ($\text{inc}-\overline{\text{inc}}$) before interacting makes the coefficient on `p401k` the effect at the mean income, without changing anything else in the model.

</details>

---

## Step 5 — Robustness: is the result driven by a few very rich households?

Step 0 showed that the top 1% hold about a quarter of all financial wealth. Since OLS minimizes squared residuals, these households might weigh heavily on the estimate.

**Task 5.1.** Re-estimate the Step 3 model after removing the 1% lowest and the 1% highest values of `nettfa`.

In [ ]:
low, high = df['nettfa'].quantile([0.01, 0.99])

# TODO: keep only households with low < nettfa < high
df_trim = ...

m5 = smf.ols('nettfa ~ p401k + ' + controls, data=df_trim).fit()
print("N =", int(m5.nobs))
print("p401k:", round(m5.params['p401k'], 2), " SE:", round(m5.bse['p401k'], 2))
assert int(m5.nobs) == 9089, "Check your filter (strict inequalities on both sides)."
print("Self-check passed!")

**Questions.**
1. Does the conclusion change? Compare the coefficient and its standard error with Step 3.
2. Why does the standard error fall so much? (Recall the variance formula of Session 2.)
3. Should the trimmed estimate replace the full-sample one?

> *Your answer:* ...

<details>
<summary>Show expected results</summary>

1. The coefficient goes from $15.84$ to $13.60$: same sign, similar magnitude, still very significant. The conclusion is **robust** to the most extreme observations.
2. The standard error is halved ($1.38\to0.68$). The variance of $\hat\beta$ is proportional to $\sigma^2$, the variance of the error. The extreme households have huge residuals, which inflate $\hat\sigma^2$. Removing them makes the model fit the remaining data much more tightly.
3. Not necessarily. Trimming is a **judgment call** (as in TD 1): very rich households are real, and their behaviour is part of the question. Removing observations based on the value of $y$ can itself distort the estimate. The right practice is to report the main estimate and the robustness check side by side: if they tell the same story, the result is more credible.

</details>

---

## Step 6 — Who chooses to participate? A binary outcome

Steps 1-5 treated participation as given. But participation is a **choice**, and understanding who makes it tells us what kind of households participants are. The outcome `p401k` is binary, so we change model (Session 4).

We restrict the sample to **eligible** households: the ineligible cannot participate, so for them `p401k` is $0$ by construction and says nothing about their choice.

**Task 6.1.** Estimate a linear probability model (LPM) of participation, and count how many predicted probabilities fall outside $[0,1]$.

In [ ]:
elig = df[df['e401k'] == 1].copy()
print("Eligible households:", len(elig), "| participation rate:", round(elig['p401k'].mean(), 3))

f_part = 'p401k ~ inc + age + marr + fsize + male'
lpm = smf.ols(f_part, data=elig).fit()
print(lpm.params.round(4))

# TODO: count the predicted probabilities above 1 and below 0
n_above_1 = ...
n_below_0 = ...
print("Predictions > 1:", n_above_1, "| < 0:", n_below_0, "| max:", round(lpm.fittedvalues.max(), 3))
assert n_above_1 == 12 and n_below_0 == 0, "Use lpm.fittedvalues."
print("Self-check passed!")

**Task 6.2.** Estimate the same model by probit and logit, then compare the coefficients, the average marginal effects, and the predicted probabilities for a married household of 3, of mean age, at different incomes.

In [ ]:
probit = smf.probit(f_part, data=elig).fit(disp=0)
logit = smf.logit(f_part, data=elig).fit(disp=0)

print(pd.DataFrame({'LPM': lpm.params, 'Probit': probit.params, 'Logit': logit.params}).round(4))
print("\nAverage marginal effects (probit):")
print(probit.get_margeff().summary_frame().round(4))
print("\nAverage marginal effects (logit):")
print(logit.get_margeff().summary_frame().round(4))
print("\nPseudo R-squared: probit", round(probit.prsquared, 3), "| logit", round(logit.prsquared, 3))

profile = pd.DataFrame({'inc': [15, 40, 80, 150], 'age': elig['age'].mean(), 'marr': 1, 'fsize': 3, 'male': 0})
profile['LPM'] = lpm.predict(profile)
profile['Probit'] = probit.predict(profile)
profile['Logit'] = logit.predict(profile)
profile[['inc', 'LPM', 'Probit', 'Logit']].round(3)

**Questions.**
1. Why is the LPM problematic here, even though most of its predictions look reasonable?
2. The probit and logit coefficients on `inc` differ a lot. Does this mean the two models disagree? What should we compare instead?
3. Interpret the marginal effect of income. Which other variables matter?
4. The pseudo $R^2$ is about $0.02$. What does this tell us about the *reasons* why eligible households participate or not? Why is this worrying for the estimates of Steps 1-5?

> *Your answer:* ...

<details>
<summary>Show expected results</summary>

Among the $3\,637$ eligible households, $70.4\%$ participate.

1. The LPM predicts probabilities above $1$ for 12 households (up to $1.10$), which is meaningless. More fundamentally, it imposes a constant effect of income on the probability, whereas a probability must flatten as it approaches 1. The table of predictions shows it: at $150$ thousand dollars, the LPM predicts $0.98$, the probit $0.93$ and the logit $0.92$. The three models agree in the middle of the distribution and diverge in the tails.
2. No. Probit and logit coefficients are on different scales (the logistic distribution has a larger variance, Session 4), so logit coefficients are typically about $1.6$ times the probit ones ($0.0150$ against $0.0088$ for income). What can be compared are the **marginal effects** on the probability, and these are practically identical: $0.0030$ for income in both models, and in the LPM ($0.0027$).
3. Ten thousand dollars more of income raise the probability of participating by about **3 percentage points**. Age, marital status, family size and gender are not significant: among eligible households, observable characteristics other than income say little about who participates.
4. Observable characteristics explain almost nothing of the decision to participate. Participation is mostly driven by things we **do not observe**: taste for saving, patience, financial literacy, attention to retirement. These very characteristics plausibly also raise financial wealth *directly*. If so, participants would have been richer in financial assets even without the plan, and the OLS estimates of Steps 1-5 still contain a selection bias. This is the endogeneity problem of Session 6.

</details>

---

## Step 7 — Is participation endogenous? An instrumental variable

### 7.1 Two stories behind the same estimate

The OLS estimate of Step 3 ($15.84$) is compatible with two causal mechanisms:

- **Story A: participation causes wealth.** The tax advantage, the employer match and the automatic deduction from pay make households save more than they otherwise would.
- **Story B: the taste for saving causes both.** Households that like to save are more likely to participate, *and* would accumulate more assets anyway.

$$
\begin{array}{ccc}
 & \text{taste for saving (unobserved, in } \varepsilon) & \\
 \swarrow & & \searrow \\
 \text{p401k} & \xrightarrow{\qquad\beta_1\qquad} & \text{nettfa}
\end{array}
$$

Under Story B, $\text{Cov}(\text{p401k}_i,\varepsilon_i)>0$ and OLS **overstates** the effect of participation, however many observable controls we add.

### 7.2 Eligibility as an instrument

The natural instrument is eligibility, `e401k`:

- **Relevance.** Participation requires eligibility: no ineligible household participates, while $70\%$ of eligible ones do. Eligibility moves participation massively.
- **Exogeneity (exclusion).** Eligibility is decided by the *employer*, not the household. The argument (Poterba, Venti and Wise) is that, once income is controlled for, whether one's employer offers a plan is unrelated to one's taste for saving, and affects wealth *only through* participation.

$$
\begin{array}{ccccc}
 & & & \text{taste for saving (in } \varepsilon) & \\
 & & \swarrow & & \searrow \\
 \text{e401k} & \xrightarrow{\ \text{relevance}\ } & \text{p401k} & \xrightarrow{\qquad\beta_1\qquad} & \text{nettfa}
\end{array}
$$

The exogeneity assumption is debatable, and it was debated. Workers who like to save may *choose* employers offering good pension plans (Engen, Gale and Scholz). In that case eligibility would itself be correlated with $\varepsilon$, and the IV estimate would also be biased upward. Keep this in mind when interpreting the result.

**Task 7.1.** Estimate the first stage (with all the controls of Step 3), and compute the $F$-statistic of the instrument.

In [ ]:
first_stage = smf.ols('p401k ~ e401k + ' + controls, data=df).fit()
print("First stage, coefficient on e401k:", round(first_stage.params['e401k'], 3))

# TODO: with a single instrument, its F-statistic is the square of its t-statistic
F_instr = ...
print("F-statistic of the instrument:", round(F_instr, 1))
assert F_instr > 10_000, "Use first_stage.tvalues['e401k']."
print("Self-check passed!")

**Task 7.2.** Estimate the reduced form (wealth on eligibility), then the second stage of 2SLS by hand, and check that 2SLS equals the ratio *reduced form / first stage* (Session 6, Section 2.3).

In [ ]:
reduced_form = smf.ols('nettfa ~ e401k + ' + controls, data=df).fit()
print("Reduced form, coefficient on e401k:", round(reduced_form.params['e401k'], 3))

# TODO: store the first-stage fitted values in df['p401k_hat']
df['p401k_hat'] = ...

second_stage = smf.ols('nettfa ~ p401k_hat + ' + controls, data=df).fit()
b_2sls = second_stage.params['p401k_hat']
print("2SLS:", round(b_2sls, 2), "| ratio reduced form / first stage:",
      round(reduced_form.params['e401k'] / first_stage.params['e401k'], 2))
assert np.isclose(b_2sls, reduced_form.params['e401k'] / first_stage.params['e401k']), "Check the fitted values."
print("Self-check passed!")

**Questions.**
1. Interpret the first-stage coefficient. Is there any risk of a weak instrument?
2. Interpret the reduced form. Why could it be the most relevant number for a policymaker deciding whether to require firms to offer a plan?
3. Compare the 2SLS estimate with OLS (Step 3). Is the direction of the change consistent with Story B? Is the difference large?
4. What exactly does the 2SLS estimate measure, and what are its main limitations?

> *Your answer:* ...

<details>
<summary>Show expected results</summary>

1. Being eligible raises the probability of participating by **69 percentage points** ($0.689$), other things equal. The $F$-statistic is about $11\,700$, far above the threshold of 10: there is no weak-instrument problem here, which is rare in applied work.
2. Eligible households hold on average **$9.4$ thousand dollars** more of financial assets than comparable ineligible ones. This is the effect of *offering* a plan, whether or not the household takes it up (an "intention-to-treat" effect). A policymaker cannot force households to participate, only firms to offer plans, so this is the effect of the policy lever they actually control.
3. 2SLS gives $9.41/0.689\approx13.66$ (standard error about $1.85$), against $15.84$ for OLS. The IV estimate is **smaller**, as Story B predicts: part of the OLS estimate was self-selection of savers into participation. But the difference is modest compared with the standard errors (the confidence intervals overlap widely). Once income and age are controlled for, selection on the taste for saving seems to explain only a small part of the participants' extra wealth.
4. It measures the effect of participation for households who participate *because* they are eligible (a local effect, Session 6), under the assumption that eligibility is exogenous given the controls. Limitations: (i) if savers choose employers that offer plans, eligibility is not exogenous and the IV estimate is still too high. (ii) `nettfa` *includes* the 401(k) balance, so a positive effect does not tell us whether other assets were reduced (the "reshuffling" question of Step 0): answering it would require separating 401(k) assets from other assets. (iii) The standard error of the by-hand second stage is not exactly the correct 2SLS one (Session 6, Section 3.3). Here the correction is negligible ($1.855$ against $1.847$), but this cannot be taken for granted.

</details>

---

## Step 8 — Synthesis

**Task 8.1.** Run the cell below to gather the estimates of the effect of participation obtained at each step.

In [ ]:
summary = pd.DataFrame({
    'Model': ['1. Simple OLS', '2. + income', '3. + functional form and controls',
              '5. Step 3, trimmed 1% tails', '7. 2SLS (instrument: e401k)'],
    'Effect of p401k': [m1.params['p401k'], m2.params['p401k'], m3.params['p401k'],
                        m5.params['p401k'], second_stage.params['p401k_hat']],
    'Std. error': [m1.bse['p401k'], m2.bse['p401k'], m3.bse['p401k'],
                   m5.bse['p401k'], second_stage.bse['p401k_hat']],
    'N': [m1.nobs, m2.nobs, m3.nobs, m5.nobs, second_stage.nobs],
})
summary.round(2)

**Task 8.2.** Write a conclusion of 8 to 10 lines that answers the question of Step 0, gives your preferred estimate with its uncertainty, and states two limitations.

> *Your answer:* ...

<details>
<summary>Show expected results</summary>

| Model | Effect of `p401k` | Std. error |
|---|---|---|
| 1. Simple OLS | $26.81$ | $1.46$ |
| 2. + income | $13.18$ | $1.42$ |
| 3. + functional form and controls | $15.84$ | $1.38$ |
| 4. Interaction (at incomes 20 / 39.3 / 80) | $12.0$ / $15.0$ / $21.1$ | |
| 5. Step 3, trimmed 1% tails | $13.60$ | $0.68$ |
| 7. 2SLS (instrument: `e401k`) | $13.66$ | $\approx1.85$ |

*Example of conclusion.* Participants in a 401(k) plan hold about 27 thousand dollars more net financial assets than non-participants, but half of this gap reflects their higher incomes. Once income, age and family characteristics are controlled for, with a flexible functional form, the gap is about 16 thousand dollars. It is robust to the exclusion of extreme values, and larger for higher-income households. Since participation is a choice that observable characteristics explain very poorly, we instrument it with eligibility, which is decided by employers. The 2SLS estimate, about 14 thousand dollars (95% confidence interval roughly $[10, 17]$), is slightly below OLS, consistent with a modest self-selection of savers into the plans. The two main limitations are that eligibility may not be fully exogenous if savers choose their employers, and that financial assets include the 401(k) balance itself, so we cannot tell whether the plans crowd out other savings.

</details>

### What we did, in general terms

The same sequence applies to any empirical question, including the one you will choose for the final assessment:

1. **Question and expected sign**, before looking at the data.
2. **Describe the data**: units, skewness, negative values (can we take logs?), outliers, missing values.
3. **Naive regression** as a benchmark, *not* as an answer.
4. **Confounders**: compare groups, add controls, predict and check the direction of the omitted variable bias.
5. **Functional form**: look at binned means, add quadratic terms or logs when the data call for them, and interpret coefficients accordingly.
6. **Heterogeneity**: does the effect depend on another variable? (interactions)
7. **Robustness**: does the result survive reasonable changes (sample, outliers)?
8. **Adapt the model to the variable**: a binary outcome calls for probit/logit and marginal effects.
9. **Endogeneity**: if an unobserved factor drives both $x$ and $y$, look for an instrument and *argue* for its validity.
10. **Synthesis**: one table, one preferred estimate, its uncertainty, and its limitations.

### Your turn

Apply the same progression to another dataset. Two suggestions where every step makes sense:
- `mroz` (married women's labour supply): the wage equation `lwage ~ educ` (OLS, Steps 1-5), the participation decision `inlf` (probit/logit, Step 6), and education instrumented by parents' education `fatheduc`, `motheduc` (Step 7).
- `card` (the example of Session 6): extend it with the additional controls available in the dataset (`smsa`, `reg661`-`reg669`, `IQ`...), and check how they change the OLS and 2SLS estimates.